# - Surface Plot of Treasury Curve
###  &nbsp; &nbsp; &nbsp; From Treasury.gov

In [1]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import requests
from io import StringIO

In [ ]:
df = pd.DataFrame()

## List of years to fetch data
years = [2020, 2021, 2022, 2023, 2024, 2025]

for year in years:
    ## Construct URL for each year
    url = f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve&field_tdr_date_value={year}"
    
    try:
        ## Fetch the page content with SSL verification
        response = requests.get(url, verify=True)
        
        ## Check if the request was successful
        if response.status_code == 200:
            ## Read HTML tables into a list of DataFrames
            tables = pd.read_html(StringIO(response.text))
            
            ## Concatenate the first table to the main DataFrame
            df = pd.concat([df, tables[0]], ignore_index=True)
            print(f"{year} Successful")
        else:
            print(f"Failed for year {year}")
    except:
        print(f"Failed to retrieve data for year {year}")
        pass

Failed to retrieve data for year 2020
Failed to retrieve data for year 2021


In [ ]:
## Reading Columns to determine what columns to keep
df.columns

In [ ]:
## Formatting DF for easier analysis

df['Date'] = pd.to_datetime(df['Date'])

df = df.drop(['20 YR', '30 YR', 'Extrapolation Factor', '6 WEEKS BANK DISCOUNT', '8 WEEKS BANK DISCOUNT',
             'COUPON EQUIVALENT', '17 WEEKS BANK DISCOUNT', 'COUPON EQUIVALENT.1',
             '52 WEEKS BANK DISCOUNT','COUPON EQUIVALENT.2', 'COUPON EQUIVALENT.3', '1.5 Mo'],axis=1).set_index('Date')

In [ ]:
## Searching for bad numbers
df[df.isna().any(axis=1)]

In [ ]:
## As '4 Mo' column has NAs, remove the column
# df = df.drop('4 Mo',axis=1)

In [ ]:
## Saving vars for easier usage in graph
x = df.columns
y = df.index
z = df.to_numpy()

## Draw Figure
fig = go.Figure(data=[go.Surface(x=x, y=y, z=z)])
fig.update_layout(title='Yield Curves',
                  scene = {"aspectratio": {"x": 1, "y": 1.4, "z": 0.9}},
                 autosize=False,
                 height=800, width=800)